In [76]:
import numpy as np
import random

def normalized(v):
    return v / np.linalg.norm(v)

def get_k1(e):
    up = np.array([0, 1, 1])
    t = [normalized(v) for v in e]
    d2 = [normalized(np.cross(up, v)) for v in e]
    X = 1 + np.dot(t[0], t[1])
    kb = 2 * np.cross(t[0], t[1]) / X
    k1 = 0.5 * np.dot(d2[0] + d2[1], kb)

    c = np.cross(t[1], sum(d2))

    dk1de0 = (-k1 * sum(t) + np.cross(t[1], sum(d2))) / np.linalg.norm(e[0]) / X
    # dk1de0 = (-t[0] * t[0].dot(c) + np.cross(t[1], sum(d2))) / np.linalg.norm(e[0]) / X
    # dk1de0 = (np.cross(t[1], sum(d2))) / np.linalg.norm(e[0]) / X
    return k1, dk1de0

e = []
e.append([1, 0, 0])
e.append(normalized([1, 0.1, 0]))
# e.append([1, 0, 0])
e = [np.array(v, dtype=np.float64) for v in e]

k1, dk1 = get_k1(e)
for i in range(3):
    offset = np.array([1 if i == j else 0 for j in range(3)], dtype=np.float64)
    stepsize = 1e-8
    offset = offset * stepsize
    e_new = e.copy()
    e_new[0] += offset
    k1_new, _ = get_k1(e_new)
    fd = k1_new - k1
    taylor = offset.dot(dk1)
    err = (fd - taylor) / (abs(fd) + stepsize)
    print(err, fd, taylor)

display(k1, dk1)




1.3912309881655627e-17 0.0 -1.3912309881655628e-25
-5.2809021321826525e-09 7.0798298196228515e-09 7.079829909819761e-09
0.2930429086454956 1.4159703354277298e-08 7.079873611327542e-09


-0.07044694060631695

array([-1.39123099e-17,  7.07982991e-01,  7.07987361e-01])

In [ ]:
# deprecated; new version in squeezeIPC
import importlib
from toy import sympy_pearlmutter as sp
import sympy
importlib.reload(sp)
Intermediate = sp.Intermediate

v = [sympy.Symbol(f'v{i}') for i in range(3)]
e = [Intermediate(f'e{i}').add(v[i + 1] - v[i]) for i in range(2)]
t = [Intermediate(f't{i}').add(e[i] / sp.Norm(e[i])) for i in range(2)]
d_prime = [sympy.Symbol(f"d_prime{i}") for i in range(2)]
c_bending = [sympy.Symbol(f'c_bending{i}') for i in range(2)]

chi = Intermediate(r'\chi').add(
    sp.Dot(t[0], t[1]) + 1)

kb = Intermediate('kb').add(
    sp.Cross(t[0], t[1]) / chi)

kappa = [Intermediate(f"k_{i}").add(
    sp.Dot(d_prime[i], kb)) for i in range(2)]

dki_dej = [
    [
        Intermediate(f'dk{i}_de{j}').add(
            (-kappa[i] * (t[0] + t[1]) + (-1)**j * sp.Cross(t[1 - j], d_prime[i])) / sp.Norm(e[j]) / chi)
        for j in range(2)] 
    for i in range(2)]
deb_dej = [Intermediate(f'deb_de{j}').add(sum([dki_dej[i][j] * c_bending[i] * kappa[i] for i in range(2)])) for j in range(2)]
deb_dvi = [-deb_dej[0], deb_dej[0] - deb_dej[1], deb_dej[1]]
# dk0_de0 = (-kappa[0] * (t[0] + t[1]) + sp.Cross(t[1], d_prime[0])) / sp.Norm(e[0]) / chi
# dk0_de0 = Intermediate('ans').add(dk0_de0)

def trace(inter, d=set()):
    if inter in d:
        return []
    d.add(inter)
    ans = []
    if isinstance(inter, sp.InterPartial):
        child = inter.child
        if isinstance(child, sp.Intermediate):
            exp = sp.get_partial(inter.child.exp, is_constant = lambda x: 'c_bending' in x.name).subs(inter.child.exp, inter.child)
        else:
            return []
    else:
        exp = inter.exp
    exp = exp.replace(lambda x: isinstance(x, Intermediate), lambda x: (ans.extend(trace(x, d)), x)[1])
    # print(f'{inter.name} = {inter.exp}')
    # display(inter.exp)
    # partial = sp.get_partial(inter.exp).subs(inter.exp, inter)
    # display(partial)
    # print(partial)
    # print(partial.replace(lambda x: isinstance(x, sympy.Symbol) and 'partial' in x.name, lambda x: sympy.Symbol(x.name.replace(r'\partial ', 'partial_'))))
    # display(sympy.simplify(sp.get_partial(inter.exp).subs(inter.exp, inter)))
    ans.append([inter, exp])
    return ans

ans = trace(Intermediate('ans').add(deb_dvi[1]))
# ans = trace(sp.InterPartial.build(Intermediate('ans').add(deb_dvi[1])))
# ans = trace(dk0_de0)
# ans = dk0_de0
# print(ans.exp.func, ans.exp.args)
# print(ans)

def sympy2taichi(s):
    s = s.replace(lambda x: isinstance(x, sympy.Symbol), lambda x: sympy.Symbol(x.name.replace(' ', '_').replace('\\', '_')))
    return s

from IPython.display import display, Markdown
for i in ans:
    # m = Markdown('$' + sympy.latex(i[0]) + '=' + sympy.latex(i[1]) + '$')
    # display(m)
    # print(m)
    print(' = '.join([str(sympy2taichi(j)) for j in i]))
    # display(*i)

# force = 1 / sp.Norm(e0) * (-k1 * t_tilde + sp.Cross(t1, d2_tilde))


# def out(exp):
#     exp = exp.replace(chi, sympy.Symbol(r'\chi'))
#     exp = exp.replace(t0, sympy.Symbol(r't^{i - 1}'))
#     exp = exp.replace(t1, sympy.Symbol(r't^{i}'))
#     # print(exp)
#     # exp.replace(lambda x: isinstance(x, sympy.Symbol) and x != e0 and x != e1, lambda x: 0)
#     # exp = exp.replace(sympy.Symbol(r'\partial X'), 0)
#     # exp = exp.replace(sympy.Symbol(r'\partial \tilde{d_2}'), 0)
#     # exp = exp.replace(sympy.Symbol(r'\partial \kappa_1'), 0)
#     display(exp)
# # force = 1 / sp.Norm(e0)
# # display(force)
# out(force)
# hessian = sp.get_partial(force)
# # display(hessian)
# out(hessian)
# h_chi_1 = sp.get_partial(1 / chi)
# out(h_chi_1)
# h_chi = sp.get_partial(chi)
# out(h_chi)

# display(e0 + e1)


e0 = -v0 + v1
t0 = e0/Norm(e0)
e1 = -v1 + v2
t1 = e1/Norm(e1)
_chi = Dot(t0, t1) + 1
kb = Cross(t0, t1)/_chi
k_0 = Dot(d_prime0, kb)
dk0_de1 = (-k_0*(t0 + t1) - Cross(t0, d_prime0))/(_chi*Norm(e1))
k_1 = Dot(d_prime1, kb)
dk1_de1 = (-k_1*(t0 + t1) - Cross(t0, d_prime1))/(_chi*Norm(e1))
deb_de1 = c_bending0*dk0_de1*k_0 + c_bending1*dk1_de1*k_1
dk0_de0 = (-k_0*(t0 + t1) + Cross(t1, d_prime0))/(_chi*Norm(e0))
dk1_de0 = (-k_1*(t0 + t1) + Cross(t1, d_prime1))/(_chi*Norm(e0))
deb_de0 = c_bending0*dk0_de0*k_0 + c_bending1*dk1_de0*k_1
ans = deb_de0 - deb_de1


In [193]:
class A:
    def __init__(self):
        self.ra = 'ra'
    
    def print(self):
        print(self.ra)

class B(A):
    def chi(self):
        self.ra = 'chi'
        super().print()

a = A()
a.print()

b = B()
b.chi()
b.print()

ra
chi
chi
